# Vertex AI Model Deployment (Minimal)


In [ ]:
# 05_deploy_vertex.ipynb
# Course 10 — Vertex AI Deployment (Quota-Safe)

from google.colab import userdata
import os

# -------------------------------------------------------------------
# Load Project Secrets (must be enabled in Colab)
# -------------------------------------------------------------------
PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GITHUB_USER = userdata.get("GITHUB_USER")
GCS_BUCKET = userdata.get("GCS_BUCKET")
TRAINING_PREFIX = userdata.get("TRAINING_PREFIX")
REPO_NAME = userdata.get("REPO_NAME")
REGION = userdata.get("REGION")

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
CLONE_PATH = f"/content/{REPO_NAME}"

## Authenticate & Configure gcloud

In [ ]:
!gcloud auth login --quiet
!gcloud config set project $PROJECT_ID
!gcloud config set compute/region $REGION

print("Project:")
!gcloud config get-value project
print("\nUser:")
!gcloud config get-value account
print("\nRegion:")
!gcloud config get-value compute/region

## Clone Repository into Colab

In [ ]:
%cd /content
!rm -rf {REPO_NAME}
!git clone {REPO_URL}

%cd {REPO_NAME}
!pwd
!ls -lh

## Install & Initialize Vertex AI SDK

In [ ]:
!pip install --quiet google-cloud-aiplatform

from google.cloud import aiplatform

aiplatform.init(
    project=PROJECT_ID,
    location=REGION
)

print("Installed google-cloud-aiplatform")

## Create Dummy / Placeholder Model (Deployment Focus)

IMPORTANT NOTE: Before saving the model, ensure the directory exists

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

dummy_model = keras.Sequential([
    layers.Input(shape=(10,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

dummy_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Model Summarization ...\n\n")
dummy_model.summary()

## Save Model Locally (Required .keras format)

In [ ]:
# Ensure artifacts directory exists prior to saving the model
os.makedirs("artifacts", exist_ok=True)

# Define model path
LOCAL_MODEL_PATH = "artifacts/module10_dummy_model.keras"
dummy_model.save(LOCAL_MODEL_PATH)

print(f"Dummy model saved at: {LOCAL_MODEL_PATH}")

## Vertex AI Endpoint (Dry Run Only — No Quota Use)

In [ ]:
ENDPOINT_DISPLAY_NAME = "module10-dummy-endpoint"

print(
    "Dry run only. Endpoint creation would use:\n"
    f"Endpoint display name: {ENDPOINT_DISPLAY_NAME}"
)

# Example (INTENTIONALLY NOT EXECUTED):
# endpoint = aiplatform.Endpoint.create(display_name=ENDPOINT_DISPLAY_NAME)

## Local Prediction Test (Mock)

In [ ]:
# Create dummy input
X_test = np.random.rand(5, 10)

# Mock prediction
predictions = dummy_model.predict(X_test)
print("Dummy predictions:", predictions)


## Deferred / Non-Blocking Notes

NOTE: Out of scope for Course 10 completion

**Next Steps:**
- Replace dummy_model with real model once trained
- Deploy to Vertex AI endpoint when quota allows
- Send real requests to endpoint
- Record output/logs in docs/course-10-artifact.md
